# Explainability: Grad-CAM and AAL2 region ranking

Runs the native 3D Grad-CAM (`multimodal_ad.models.gradcam`) on a tiny synthetic volume/model, then the AAL2 region-ranking utility (`multimodal_ad.models.regions`) on a small toy atlas, since the real AAL2 atlas is not shipped with this repository. Requires the `model` extra: `uv sync --extra model`.

In [ ]:
import numpy as np
import pandas as pd

from multimodal_ad.models.architecture import Cnn3DConfig, build_3d_cnn
from multimodal_ad.models.gradcam import (
    last_conv_layer_name,
    make_gradcam_heatmap,
    unwrap_output_activation,
)
from multimodal_ad.models.regions import rank_regions

## Native Grad-CAM on a tiny synthetic volume

Same reduced-filter model shape as `02-tiny-model-workflow.ipynb`. Grad-CAM needs raw logits, not a squashed sigmoid output, hence `unwrap_output_activation`.

In [ ]:
SIZE = 48
rng = np.random.default_rng(1234)

config = Cnn3DConfig(
    width=SIZE, height=SIZE, depth=SIZE, filters=(4, 4, 8, 8), dense_units=16,
    name="tiny-gradcam",
)
model = build_3d_cnn(config)
unwrap_output_activation(model)

layer_name = last_conv_layer_name(model)
volume = rng.random((1, SIZE, SIZE, SIZE, 1)).astype("float32")
heatmap = make_gradcam_heatmap(volume, model, layer_name)
heatmap.shape, float(heatmap.min()), float(heatmap.max())

## AAL2 region ranking: how you supply the real atlas

`rank_regions` needs an AAL2 atlas volume (`atlas.nii.gz`), which is **not** checked into this repository (only `AAL2_Atlas_Labels.csv`, the region name -> intensity mapping, is). To use your own atlas:

```python
import nibabel as nib
from multimodal_ad.models.regions import (
    ATLAS_PLACEMENT, HEATMAP_PLACEMENT, load_region_labels, pad_to_frame,
)

atlas = np.asarray(nib.load("path/to/atlas.nii.gz").get_fdata())
atlas = pad_to_frame(atlas, ATLAS_PLACEMENT)
heatmap = pad_to_frame(heatmap, HEATMAP_PLACEMENT)
region_labels = load_region_labels("AAL2_Atlas_Labels.csv")
ranking = rank_regions(atlas, region_labels, {"pos": heatmap})
```

To keep this notebook self-contained and OASIS-free, we use a tiny **toy** atlas with two synthetic regions instead of the real `(91, 109, 91)` AAL2 volume.

In [ ]:
toy_atlas = np.zeros((4, 4, 4), dtype=np.float64)
toy_atlas[0, 0, 0] = 10.0  # stands in for one AAL2 region
toy_atlas[1, 1, 1] = 20.0  # stands in for another

toy_heatmap = np.zeros((4, 4, 4), dtype=np.float64)
toy_heatmap[0, 0, 0] = 0.9
toy_heatmap[1, 1, 1] = 0.2

region_labels = pd.DataFrame({"intensity": [10.0, 20.0]}, index=["region_a", "region_b"])
region_labels.index.name = "name"

ranking = rank_regions(toy_atlas, region_labels, {"toy": toy_heatmap})
ranking

## Legacy `mean` vs. corrected `region mean`

`rank_regions` reports two different mean columns (see `multimodal_ad.models.regions` module docstring for the full derivation):

- `"{name} mean"`: the **legacy notebook's exact formula** (`exploration.ipynb`), region sum divided by the *entire common-frame voxel count* (here, all 64 voxels), not the region's own size. Kept for reproducing the published paper's region tables; it shrinks toward zero for small regions purely from a huge, mostly-zero denominator, not from lower Grad-CAM importance.
- `"{name} region mean"`: the **corrected**, size-comparable mean, region sum divided by the region's *own* voxel count. This is what "mean Grad-CAM in region X" should mean, and is what new analysis should use.

In [ ]:
ranking[["part", "toy mean", "toy region mean"]]

## Next steps

- Legacy region-ranking notebook: `notebooks/legacy/exploration-region-ranking.ipynb`.
- Open ambiguities in the atlas/heatmap spatial alignment are documented in `docs/legacy-notebooks-inventory.md`.